# Neural Network

In [1]:
import random
random.seed(42)

In [2]:
!pip install datasets nltk staticvectors scikit-learn torch -q

## Loading IMDB dataset

In [3]:
import pandas as pd
from datasets import load_dataset

In [4]:
dataset = load_dataset('imdb')

In [5]:
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})

In [6]:
train = dataset['train'].shuffle(seed=42).select(range(5000))
test = dataset['test'].shuffle(seed=42).select(range(1000))

In [7]:
train = pd.DataFrame(train)
test = pd.DataFrame(test)

In [8]:
train.head()

,text,label
0,There is no relation at all between Fortier an...,1
1,This movie is a great. The plot is very true t...,1
2,"George P. Cosmatos' ""Rambo: First Blood Part I...",0
3,In the process of trying to establish the audi...,1
4,"Yeh, I know -- you're quivering with excitemen...",0


## Preprocessing

In [9]:
import nltk, re

In [10]:
def preprocess(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    tokens = nltk.word_tokenize(text)
    return tokens

In [11]:
train['tokens'] = train['text'].apply(preprocess)
test['tokens'] = test['text'].apply(preprocess)

In [12]:
train.head()

,text,label,tokens
0,There is no relation at all between Fortier an...,1,"[there, is, no, relation, at, all, between, fo..."
1,This movie is a great. The plot is very true t...,1,"[this, movie, is, a, great, the, plot, is, ver..."
2,"George P. Cosmatos' ""Rambo: First Blood Part I...",0,"[george, p, cosmatos, rambo, first, blood, par..."
3,In the process of trying to establish the audi...,1,"[in, the, process, of, trying, to, establish, ..."
4,"Yeh, I know -- you're quivering with excitemen...",0,"[yeh, i, know, youre, quivering, with, excitem..."


## Using GloVe

In [13]:
from staticvectors import StaticVectors

In [14]:
model = StaticVectors('neuml/glove-6B')

In [15]:
text = ['this', 'is', 'the', 'best', 'scifi', 'that', 'i', 'have', 'seen', 'in', 'my', 'years', 'of', 'watching', 'scifi', 'i', 'also', 'believe', 'that', 'dark', 'angel', 'will', 'become', 'a', 'cult', 'favorite', 'the', 'action', 'is', 'great', 'but', 'jessica', 'alba', 'is', 'the', 'best', 'and', 'most', 'gorgeous', 'star', 'on', 'tv', 'today']

In [16]:
len(text)

43

In [17]:
testing = model.embeddings(text)

In [18]:
testing.shape

(43, 300)

## Sentence Embedding

In [19]:
import numpy as np

In [20]:
def sentence_embedding(tokens, model, dim=300):
    try:
        vectors = model.embeddings(tokens)
        return np.mean(vectors, axis=0)
    except:
        return np.zeros(dim)

In [21]:
vectors = sentence_embedding(text, model)
vectors.shape

(300,)

## Training and testing datasets

In [22]:
X_train = np.array([sentence_embedding(tokens, model, 300) for tokens in train['tokens']])
y_train = train['label'].values

In [23]:
print(f'{X_train.shape} {y_train.shape}')

(5000, 300) (5000,)


In [24]:
X_test = np.array([sentence_embedding(tokens, model, 300) for tokens in test['tokens']])
y_test = test['label'].values

In [25]:
print(f'{X_test.shape} {y_test.shape}')

(1000, 300) (1000,)


## Device

In [26]:
import torch

In [27]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

cuda


## Training and testing to Torch

In [28]:
X_train_t = torch.tensor(X_train, dtype=torch.float32)
X_test_t  = torch.tensor(X_test, dtype=torch.float32)

In [29]:
y_train_t = torch.tensor(y_train, dtype=torch.long)
y_test_t  = torch.tensor(y_test, dtype=torch.long)

## Neural Network Classifier

In [30]:
import torch.nn as nn
import torch.optim as optim

In [31]:
class LNN(nn.Module):

    def __init__(self):
        super().__init__()

        self.w = nn.Linear(300, 512)
        self.relu = nn.ReLU()
        self.u = nn.Linear(512, 2)
    
    def forward(self, x):
        z1 = self.w(x)
        a1 = self.relu(z1)
        z2 = self.u(a1)
        return z2

In [32]:
model = LNN()
print(model)

LNN(
  (w): Linear(in_features=300, out_features=512, bias=True)
  (relu): ReLU()
  (u): Linear(in_features=512, out_features=2, bias=True)
)


In [33]:
epochs = 100
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

## Training

In [34]:
for epoch in range(epochs):
    model.train()
    
    outputs = model(X_train_t)
    loss = criterion(outputs, y_train_t)
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    if (epoch+1) % 5 == 0:
        print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

Epoch 5, Loss: 0.6894
Epoch 10, Loss: 0.6833
Epoch 15, Loss: 0.6746
Epoch 20, Loss: 0.6631
Epoch 25, Loss: 0.6492
Epoch 30, Loss: 0.6334
Epoch 35, Loss: 0.6162
Epoch 40, Loss: 0.5986
Epoch 45, Loss: 0.5811
Epoch 50, Loss: 0.5638
Epoch 55, Loss: 0.5468
Epoch 60, Loss: 0.5305
Epoch 65, Loss: 0.5150
Epoch 70, Loss: 0.5007
Epoch 75, Loss: 0.4879
Epoch 80, Loss: 0.4764
Epoch 85, Loss: 0.4662
Epoch 90, Loss: 0.4571
Epoch 95, Loss: 0.4491
Epoch 100, Loss: 0.4419


## Evaluating

In [35]:
from sklearn.metrics import classification_report

In [36]:
model.eval()

with torch.no_grad():
    outputs = model(X_test_t)
    _, y_pred = torch.max(outputs, 1)

y_pred = y_pred.numpy()

In [37]:
y_pred[:10]

array([1, 0, 0, 0, 0, 1, 1, 0, 0, 1])

In [38]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.78      0.81      0.79       512
           1       0.79      0.76      0.78       488

    accuracy                           0.79      1000
   macro avg       0.79      0.79      0.79      1000
weighted avg       0.79      0.79      0.79      1000

